In [ ]:
                                      # "RIDE DEMAND FORECASTING DATA PREP ENGINE"

In [2]:
# step 1: import libraties

import pandas as pd
import numpy as np
import sqlite3

from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer

from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler
)

from scipy.stats import zscore

import warnings
warnings.filterwarnings("ignore")

In [17]:
# step 2: load dataset
# csv load
riders = pd.read_excel("riders.xlsx")
riders.head()

,rider_id,name,age,gender,city,signup_date,total_rides,cancelled_rides,avg_rating
0,R0001,Aarav Das,23,Male,Pune,2020-06-29,56,0,3.76
1,R0002,Ishaan Nair,39,Female,Mumbai,2019-11-23,70,5,4.12
2,R0003,Kavya Reddy,34,Male,Pune,2023-05-04,45,9,3.76
3,R0004,Aarav Nair,19,Other,Kolkata,2019-07-28,464,5,3.19
4,R0005,Diya Reddy,27,Male,Ahmedabad,2021-05-31,294,30,3.53


In [16]:
# trips JSON
trips = pd.read_json("trips.json")
trips.head()

,trip_id,rider_id,zone,distance_km,duration_min,fare_amount,payment_mode,ride_date,surge_flag
0,T00001,R0037,Zone_10,11.83,74.59,104.88,Cash,2023-11-13,0
1,T00002,R0104,Zone_9,3.86,35.59,40.48,Cash,2023-07-28,1
2,T00003,R0045,Zone_8,4.70,31.03,46.39,Cash,2024-01-14,1
3,T00004,R0089,Zone_2,11.06,59.48,257.64,Cash,2023-12-13,0
4,T00005,R0003,Zone_5,7.28,67.59,72.74,UPI,2023-03-15,1


In [124]:
# SQL load
conn = sqlite3.connect("city_zones.db")

cursor = conn.cursor()

with open("city_zones.sql", "r") as file:
    sql_script = file.read()

cursor.executescript(sql_script)

zones = pd.read_sql(
    "SELECT * FROM city_zones",
    conn
)

zones.head()

,zone_name,population_density,traffic_index,avg_speed_kmph,zone_type
0,Zone_1,4921,2.43,30.9,Residential
1,Zone_2,6371,0.91,58.4,Residential
2,Zone_3,12971,2.11,38.0,Business
3,Zone_4,4038,2.46,48.2,Business
4,Zone_5,2590,1.31,43.9,Business


In [19]:
# step 3: data understanding
# first 5 rows
print(riders.head())
print(trips.head())
print(zones.head())

  rider_id         name  age  gender       city signup_date  total_rides  \
0    R0001    Aarav Das   23    Male       Pune  2020-06-29           56   
1    R0002  Ishaan Nair   39  Female     Mumbai  2019-11-23           70   
2    R0003  Kavya Reddy   34    Male       Pune  2023-05-04           45   
3    R0004   Aarav Nair   19   Other    Kolkata  2019-07-28          464   
4    R0005   Diya Reddy   27    Male  Ahmedabad  2021-05-31          294   

   cancelled_rides  avg_rating  
0                0        3.76  
1                5        4.12  
2                9        3.76  
3                5        3.19  
4               30        3.53  
  trip_id rider_id     zone  distance_km  duration_min  fare_amount  \
0  T00001    R0037  Zone_10        11.83         74.59       104.88   
1  T00002    R0104   Zone_9         3.86         35.59        40.48   
2  T00003    R0045   Zone_8         4.70         31.03        46.39   
3  T00004    R0089   Zone_2        11.06         59.48       

In [20]:
# dataset information
riders.info()
trips.info()
zones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   rider_id         300 non-null    object        
 1   name             300 non-null    object        
 2   age              300 non-null    int64         
 3   gender           300 non-null    object        
 4   city             300 non-null    object        
 5   signup_date      300 non-null    datetime64[ns]
 6   total_rides      300 non-null    int64         
 7   cancelled_rides  300 non-null    int64         
 8   avg_rating       300 non-null    float64       
dtypes: datetime64[ns](1), float64(1), int64(3), object(4)
memory usage: 21.2+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   trip_id       2000 non-null   obje

In [21]:
# missing value
riders.isnull().sum()
trips.isnull().sum()
zones.isnull().sum()

zone_name             0
population_density    0
traffic_index         0
avg_speed_kmph        0
zone_type             0
dtype: int64

In [22]:
# duplicate value
riders.duplicated().sum()
trips.duplicated().sum()
zones.duplicated().sum()

np.int64(0)

In [25]:
# remove duplicates
riders.drop_duplicates(inplace=True)
trips.drop_duplicates(inplace=True)
zones.drop_duplicates(inplace=True)

In [34]:
# step 4: handle missing value
# numerical columns

num_imputer = SimpleImputer(strategy="mean")

riders["age"] = num_imputer.fit_transform(
    riders[["age"]]
)

riders["age"].head()

0    23.0
1    39.0
2    34.0
3    19.0
4    27.0
Name: age, dtype: float64

In [35]:
# most frequent strategy
cat_imputer = SimpleImputer(
    strategy="most_frequent"
)

riders["gender"] = cat_imputer.fit_transform(
    riders[["gender"]]
).ravel()

riders["gender"].head()

0      Male
1    Female
2      Male
3     Other
4      Male
Name: gender, dtype: object

In [42]:
# knn imputer

knn = KNNImputer(n_neighbors=5)

trips[
[
"distance_km",
"duration_min",
"fare_amount"
]
] = knn.fit_transform(
trips[
[
"distance_km",
"duration_min",
"fare_amount"
]
]
)


In [37]:
print(trips.columns.tolist())

['trip_id', 'rider_id', 'zone', 'distance_km', 'duration_min', 'fare_amount', 'payment_mode', 'ride_date', 'surge_flag']


In [51]:
# step 5: date conversion
trips["ride_date"] = pd.to_datetime(
    trips["ride_date"]
)
trips["ride_date"].head()

0   2023-11-13
1   2023-07-28
2   2024-01-14
3   2023-12-13
4   2023-03-15
Name: ride_date, dtype: datetime64[ns]

In [55]:
# step 6: remove invalid records
# negative fare

trips = trips[
trips["fare_amount"] >=0
] 
trips["fare_amount"].head()

0    104.88
1     40.48
2     46.39
3    257.64
4     72.74
Name: fare_amount, dtype: float64

In [56]:
# Negative Distance
trips = trips[
trips["distance_km"] >= 0
]
trips["distance_km"].head()

0    11.83
1     3.86
2     4.70
3    11.06
4     7.28
Name: distance_km, dtype: float64

In [64]:
# outlier handling
# Z - Score method
trips["fare_z"] = zscore(
    trips["fare_amount"]
)

trips["distance_z"] = zscore(
    trips["distance_km"]
)

trips = trips[
    (trips["fare_z"].abs() < 3)
    &
    (trips["distance_z"].abs() < 3)
]
print("After Z-score filtering:", trips.shape)

After Z-score filtering: (1962, 11)


In [60]:
# IQR method
Q1 = trips["duration_min"].quantile(0.25)

Q3 = trips["duration_min"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR

upper = Q3 + 1.5 * IQR

trips = trips[
    (trips["duration_min"] >= lower)
    &
    (trips["duration_min"] <= upper)
]
print("After IQR filtering:", trips.shape)

After IQR filtering: (1963, 11)


In [65]:
# winsorization
trips['fare_amount'] = np.where(trips['fare_amount'] > trips['fare_amount'].quantile(0.95),
                                trips['fare_amount'].quantile(0.95),
                                trips['fare_amount'])
print("After Winsorization, fare stats:")
print(trips['fare_amount'].describe())

After Winsorization, fare stats:
count    1962.000000
mean      128.614309
std        75.846205
min         0.250000
25%        69.715000
50%       120.065000
75%       178.722500
max       279.816500
Name: fare_amount, dtype: float64


In [69]:
# step 8: datetime features
trips["hour"] = trips["ride_date"].dt.hour
trips["day_of_week"] = trips["ride_date"].dt.day_name()
trips["month"] = trips["ride_date"].dt.month

print(trips[['ride_date','hour','day_of_week','month']].head(10))


   ride_date  hour day_of_week  month
0 2023-11-13     0      Monday     11
1 2023-07-28     0      Friday      7
2 2024-01-14     0      Sunday      1
3 2023-12-13     0   Wednesday     12
4 2023-03-15     0   Wednesday      3
5 2024-02-10     0    Saturday      2
6 2023-08-21     0      Monday      8
7 2024-03-10     0      Sunday      3
8 2024-05-20     0      Monday      5
9 2023-03-01     0   Wednesday      3


In [72]:
# step 9 : encoding
# label encoding
le = LabelEncoder()

riders["gender"] = le.fit_transform(
riders["gender"]
)
print(riders[['gender']].head())

   gender
0       1
1       0
2       1
3       2
4       1


In [90]:
print(trips.head())
print("New columns:", trips.columns)


  trip_id rider_id     zone  distance_km  duration_min  fare_amount  \
0  T00001    R0037  Zone_10        11.83         74.59       104.88   
1  T00002    R0104   Zone_9         3.86         35.59        40.48   
2  T00003    R0045   Zone_8         4.70         31.03        46.39   
3  T00004    R0089   Zone_2        11.06         59.48       257.64   
4  T00005    R0003   Zone_5         7.28         67.59        72.74   

   ride_date  surge_flag    fare_z  distance_z  hour day_of_week  month  \
0 2023-11-13           0 -0.318757    0.888795     0      Monday     11   
1 2023-07-28           1 -1.131616   -0.985797     0      Friday      7   
2 2024-01-14           1 -1.057020   -0.788224     0      Sunday      1   
3 2023-12-13           0  1.609384    0.707686     0   Wednesday     12   
4 2023-03-15           1 -0.724429   -0.181393     0   Wednesday      3   

   payment_mode_Cash  payment_mode_Credit Card  payment_mode_UPI  \
0               True                     False        

In [94]:
# ordinal encoding
traffic_map = {
"Low":1,
"Medium":2,
"High":3
}

zones["traffic_index"] = (
zones["traffic_index"]
.map(traffic_map)
)
print(zones[['traffic_index']].head())


   traffic_index
0            NaN
1            NaN
2            NaN
3            NaN
4            NaN


In [96]:
# binning
riders["ride_frequency"] = pd.qcut(
riders["total_rides"],
q=3,
labels=[
"Low",
"Medium",
"High"
]
)
print(riders[['total_rides','ride_frequency']].head(10))


   total_rides ride_frequency
0           56            Low
1           70            Low
2           45            Low
3          464           High
4          294         Medium
5          248         Medium
6          458           High
7          215         Medium
8          172         Medium
9          173         Medium


In [102]:
# trasformation
trips["fare_amount"] = np.log1p(
trips["fare_amount"]
)
trips["distance_km"] = np.log1p(
trips["distance_km"]
)
print(trips[['fare_amount','distance_km']].head(10))

   fare_amount  distance_km
0     0.695995     2.551786
1     0.661262     1.581038
2     0.666837     1.740466
3     0.721682     2.489894
4     0.683732     2.113843
5     0.723780     2.745988
6     0.675745     1.701105
7     0.695214     2.545531
8     0.711430     2.371178
9     0.693924     2.124654


In [104]:
# square root
trips["duration_min"] = np.sqrt(
trips["duration_min"]
)

In [106]:
# step 12: feature scaling
# stadardscaler
ss = StandardScaler()

trips[
[
"fare_amount",
"distance_km"
]
] = ss.fit_transform(
trips[
[
"fare_amount",
"distance_km"
]
]
)

In [108]:
# minmaxscaler
mm = MinMaxScaler()

trips[
[
"duration_min"
]
] = mm.fit_transform(
trips[
[
"duration_min"
]
]
)

In [119]:
# average ride distance

trips['avg_ride_distance'] = trips['distance_km'] / trips['trip_id'].count()
trips['avg_ride_fare'] = trips['fare_amount'] / trips['trip_id'].count()

# Surge flag
trips['surge_flag'] = np.where(trips['fare_amount'] > 2*trips['distance_km'], 1, 0)

print(trips[['distance_km','fare_amount','avg_ride_distance','avg_ride_fare','surge_flag']].head())


   distance_km  fare_amount  avg_ride_distance  avg_ride_fare  surge_flag
0     0.814928     0.165587           0.000415       0.000084           0
1    -0.767623    -0.592240          -0.000391      -0.000302           1
2    -0.507718    -0.470602          -0.000259      -0.000240           1
3     0.714029     0.726041           0.000364       0.000370           0
4     0.100975    -0.101976           0.000051      -0.000052           0


In [114]:
# Peak hour flag
trips['is_peak_hour'] = trips['hour'].apply(lambda x: 1 if x in [7,8,9,18,19,20,21] else 0)

print(trips[['hour','is_peak_hour']].head())

# Days since signup
riders['signup_date'] = pd.to_datetime(riders['signup_date'])
riders['days_since_signup'] = (pd.to_datetime("today") - riders['signup_date']).dt.days

print(riders[['signup_date','days_since_signup']].head())


   hour  is_peak_hour
0     0             0
1     0             0
2     0             0
3     0             0
4     0             0
  signup_date  days_since_signup
0  2020-06-29               2181
1  2019-11-23               2400
2  2023-05-04               1142
3  2019-07-28               2518
4  2021-05-31               1845


In [118]:
# cancellation rate
riders['ride_cancelled_rate'] = riders['cancelled_rides'] / riders['total_rides']
print(riders[['rider_id','cancelled_rides','total_rides','ride_cancelled_rate']].head())


  rider_id  cancelled_rides  total_rides  ride_cancelled_rate
0    R0001                0           56             0.000000
1    R0002                5           70             0.071429
2    R0003                9           45             0.200000
3    R0004                5          464             0.010776
4    R0005               30          294             0.102041


In [129]:
# Merge datasets
df = trips.merge(
    riders,
    on="rider_id",
    how="left"
)

df = df.merge(
    zones,
    left_on="zone",      
    right_on="zone_name",
    how="left"
)

print(df.head())
print(df.columns)



  trip_id rider_id     zone  distance_km  duration_min  fare_amount  \
0  T00001    R0037  Zone_10     0.814928      0.677782     0.165587   
1  T00002    R0104   Zone_9    -0.767623      0.457040    -0.592240   
2  T00003    R0045   Zone_8    -0.507718      0.424370    -0.470602   
3  T00004    R0089   Zone_2     0.714029      0.601395     0.726041   
4  T00005    R0003   Zone_5     0.100975      0.643463    -0.101976   

   ride_date  surge_flag    fare_z  distance_z  ...  cancelled_rides  \
0 2023-11-13           0 -0.318757    0.888795  ...                6   
1 2023-07-28           1 -1.131616   -0.985797  ...                1   
2 2024-01-14           1 -1.057020   -0.788224  ...               16   
3 2023-12-13           0  1.609384    0.707686  ...                6   
4 2023-03-15           0 -0.724429   -0.181393  ...                9   

  avg_rating  ride_frequency  days_since_signup  ride_cancelled_rate  \
0       3.36            High               2036             0.012270

In [131]:
# final dataset
df.head()
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1962 entries, 0 to 1961
Data columns (total 36 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   trip_id                   1962 non-null   object        
 1   rider_id                  1962 non-null   object        
 2   zone                      1962 non-null   object        
 3   distance_km               1962 non-null   float64       
 4   duration_min              1962 non-null   float64       
 5   fare_amount               1962 non-null   float64       
 6   ride_date                 1962 non-null   datetime64[ns]
 7   surge_flag                1962 non-null   int64         
 8   fare_z                    1962 non-null   float64       
 9   distance_z                1962 non-null   float64       
 10  hour                      1962 non-null   int32         
 11  day_of_week               1962 non-null   object        
 12  month               

(1962, 36)

In [133]:
# export CSV
df.to_csv(
"final_prepared_rides_dataset.csv",
index=False
)

In [134]:
# summary table
summary = pd.DataFrame({

"Rows":[df.shape[0]],

"Columns":[df.shape[1]],

"Missing Values":[
df.isnull().sum().sum()
]

})

summary

,Rows,Columns,Missing Values
0,1962,36,0


In [140]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
df,
title="Ride Data EDA Report"
)

profile.to_file(
"Ride_EDA_Report.html"
)

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████████████████████████████████████████████████████████████████████████████| 36/36 [00:00<00:00, 39.28it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]